# NB1 · DUK leg kinematics

This notebook uses one canonical robot frame. **A=(0,0) only in the body-attached frame; A moves in the ground/world frame.** Every other point is produced by geometry equations.

- +Ox points right, +Oy points up, counter-clockwise is positive.
- D→A = -30°, therefore A→D is the opposite direction.
- `q` is the directed counter-clockwise angle from A→B to D→A.
- `θ_AB = θ_DA - q`.
- D is derived from AD and `θ_AD`; B is derived from AB and `θ_AB`.
- C is the physical circle-intersection solution selected by named topology.
- H/O are derived from the moving rigid BCO basis.
- Canonical geometry is unchanged. A separate rigid world placement makes the tire touch ground z=0.
- The wheel center has world height equal to its radius; the body moves as q changes. Local coordinate/Jacobian plots are labeled body-frame quantities.

[DUK v3 chapter 02, corrected 2026-11-11](https://app.notion.com/p/4131f3cc033281a2aae1f8e6bc96d1fc).
Mathematical +x has not yet been assigned to the real robot's front/back by CAD.


In [ ]:
import hashlib
import math
import subprocess
from pathlib import Path

from duk_core.modeling import (
    leg_jacobian,
    load_leg_geometry_json,
    sample_leg_trajectory,
    solve_leg_pose,
)
from duk_core.modeling.ground_contact import place_leg_on_ground
from duk_core.visualization import (
    draw_leg_linkage,
    draw_point_trajectories,
    plot_point_coordinates,
    plot_point_jacobian,
)
from duk_core.visualization.notebook import configure_notebook_plots

configure_notebook_plots()


def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "robot/config/physical/duk_leg_geometry.json").exists():
            return candidate
    raise FileNotFoundError("Run the notebook from a DUK repository checkout")


repo_root = find_repo_root()
geometry_path = repo_root / "robot/config/physical/duk_leg_geometry.json"
geometry = load_leg_geometry_json(geometry_path)
print(
    "Commit:",
    subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=repo_root, text=True
    ).strip(),
)
print(
    "Working tree dirty:",
    bool(subprocess.check_output(["git", "status", "--porcelain"], cwd=repo_root)),
)
print("Geometry SHA-256:", hashlib.sha256(geometry_path.read_bytes()).hexdigest())
print(f"DA = {math.degrees(geometry.da_angle_rad):+.1f}°")
print(f"AD = {math.degrees(geometry.ad_angle_rad):+.1f}°")
print("D is derived from A, AD and θ_AD =", geometry.d_point(), "mm")
geometry


configure_notebook_plots()

In [ ]:
lower, upper = geometry.geometric_linkage_angle_limits_rad()
print(f"BO = {geometry.bo_mm:.3f} mm")
print(f"CO = {geometry.co_mm:.3f} mm")
print(f"geometric q range = {math.degrees(lower):.3f} .. {math.degrees(upper):.3f}°")
print("q is the directed CCW angle from A→B to D→A.")
print("The endpoints are geometric singular boundaries, not operating limits.")

In [ ]:
q_deg = 75.0
q = math.radians(q_deg)
pose = solve_leg_pose(geometry, q)
jac = leg_jacobian(geometry, pose)
theta_ab_deg = math.degrees(geometry.ab_angle_rad(q))
print(f"q = {q_deg:.2f}°")
print(f"derived A→B direction = {theta_ab_deg:+.2f}° from +Ox")
print("Body-frame coordinates (not world height):")
for name in ("A", "B", "C", "D", "H", "O"):
    print(name, "=", pose.point(name), "mm")
print("dO/dq =", jac.d_o, "mm/rad")
draw_leg_linkage(pose, geometry, ground_contact=True).set_title(
    f"DUK leg on ground — q={q_deg:.1f}°"
)

## Explore one pose

Move the angle slider and switch between ground/world and body coordinates.
The table follows the selected frame; the Jacobian remains a body-frame quantity.
The 2° margin excludes geometric singularities and is only a display range.

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import HTML, display

from duk_core.modeling.kinematic_bench import sample_bench_leg
from duk_core.visualization.notebook import result_table


def inspect_pose(angle_deg, ground_frame):
    local = solve_leg_pose(geometry, math.radians(angle_deg))
    shown = place_leg_on_ground(geometry, local) if ground_frame else local
    fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)
    draw_leg_linkage(local, geometry, ax=ax, ground_contact=ground_frame)
    frame = "Ground/world" if ground_frame else "Body"
    clearance = sample_bench_leg(
        geometry, math.radians(angle_deg)
    ).projected_hip_clearance_mm
    print(
        f"Projected A–wheel clearance: {clearance:+.2f} mm (CAD collision unresolved)"
    )
    ax.set_title(f"{frame} frame · q = {angle_deg:.1f}°")
    plt.show()
    display(
        HTML(
            result_table(
                [
                    {
                        "Point": key,
                        "x (mm)": shown.point(key)[0],
                        ("z (mm)" if ground_frame else "y (mm)"): shown.point(key)[1],
                    }
                    for key in ("A", "B", "C", "D", "H", "O")
                ],
                caption=f"{frame}-frame coordinates",
            )
        )
    )


pose_controls = widgets.interactive(
    inspect_pose,
    angle_deg=widgets.FloatSlider(
        value=q_deg,
        min=math.degrees(lower) + 2,
        max=math.degrees(upper) - 2,
        step=0.5,
        description="q (deg)",
        continuous_update=False,
    ),
    ground_frame=widgets.Checkbox(value=True, description="Ground/world frame"),
)
display(pose_controls)

In [ ]:
margin = math.radians(2.0)
poses = sample_leg_trajectory(geometry, lower + margin, upper - margin, 180)
draw_point_trajectories(poses).set_title("B/C/H/O trajectories — canonical robot frame")

In [ ]:
plot_point_coordinates(poses, "O").set_title("Body-frame wheel-center coordinates vs q")

In [ ]:
plot_point_jacobian(geometry, poses, "O").set_title("Wheel-center Jacobian dO/dq")

## Ground/world placement

The coordinate, trajectory and Jacobian plots above analyze body-frame quantities; the mechanism drawing uses the ground/world frame. Ground contact translates the **entire** solved mechanism by `z_A = radius - O_local.y`; no link is scaled, mirrored or moved independently. The circle is tangent to z=0 for every q. This is normal-contact geometry; no forces or no-slip rolling are imposed.


In [ ]:
world_poses = tuple(place_leg_on_ground(geometry, p) for p in poses)
world_wheel_z = [p.o[1] for p in world_poses]
body_height = [p.a[1] for p in world_poses]
ground_error = max(abs(z - geometry.wheel_radius_mm) for z in world_wheel_z)
assert ground_error < 1e-9
print("max tire / ground gap:", ground_error, "mm")
angles_deg = [math.degrees(p.linkage_angle_rad) for p in poses]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(angles_deg, body_height, label="Body A world height")
axes[0].set_ylabel("world z (mm)")
axes[1].plot(angles_deg, world_wheel_z, label="Wheel center world z")
axes[1].axhline(0, color="0.25", label="ground")
axes[1].axhline(geometry.wheel_radius_mm, ls="--", color="orange", label="wheel radius")
for ax in axes:
    ax.set_xlabel("q (deg)")
    ax.grid(True)
    ax.legend()
plt.tight_layout()
plt.show()

## V2 geometry regression and derivative checks

The table is a rounded calculation from [DUK v2](https://app.notion.com/p/4131f3cc03328185a0f0d1ab682af2f2).
Numerical differences below are compared in mm and radians. The sweep excludes the
geometric tangencies, and the checks at those boundaries confirm that derivatives
are rejected. Continue with the [motion workbench](../motion/duk-leg-motion.ipynb)
for one solved pose per animation frame.

In [ ]:
import numpy as np

from duk_core.modeling import KinematicSingularityError, leg_second_derivative

reference = np.array(
    [
        (75, -0.468, -253.296, 95.127, 92.377),
        (90, 5.482, -223.665, -8.176, 133.299),
        (105, 2.680, -182.641, -7.092, 183.209),
        (115, 2.656, -146.509, 7.903, 235.256),
        (120, 3.726, -124.287, 16.486, 276.867),
        (130, 6.881, -61.866, 3.897, 494.117),
    ]
)
recomputed = []
for angle, *_ in reference:
    sample = solve_leg_pose(geometry, math.radians(angle))
    recomputed.append((*sample.o, *leg_jacobian(geometry, sample).d_o))
np.testing.assert_allclose(recomputed, reference[:, 1:], atol=0.00051, rtol=0)
print(
    "Max deviation from rounded V2 table:",
    np.max(abs(recomputed - reference[:, 1:]), axis=0),
)

step = 1e-5
for sample in poses:
    q_sample = sample.linkage_angle_rad
    left = solve_leg_pose(geometry, q_sample - step)
    right = solve_leg_pose(geometry, q_sample + step)
    jac_left = np.array(leg_jacobian(geometry, left).d_o)
    jac_right = np.array(leg_jacobian(geometry, right).d_o)
    np.testing.assert_allclose(
        leg_jacobian(geometry, sample).d_o,
        (np.array(right.o) - left.o) / (2 * step),
        rtol=1e-6,
        atol=1e-5,
    )
    np.testing.assert_allclose(
        leg_second_derivative(geometry, sample).dd_o,
        (jac_right - jac_left) / (2 * step),
        rtol=1e-6,
        atol=1e-4,
    )
    b, c, o = sample.bco_triangle
    np.testing.assert_allclose(
        [math.dist(b, c), math.dist(b, o), math.dist(c, o)],
        [geometry.bc_mm, geometry.bo_mm, geometry.co_mm],
        atol=1e-9,
        rtol=0,
    )

for boundary in (lower, upper):
    try:
        leg_jacobian(geometry, solve_leg_pose(geometry, boundary))
    except KinematicSingularityError:
        print(f"Derivative rejected at {math.degrees(boundary):.6f}°")
    else:
        raise AssertionError("The geometric boundary must be rejected for derivatives")

## Corrected v3 geometry checks

The assembly domain is approximately 69.0°–135.3°. Jacobians grow sharply
near its boundaries; a geometric display reserve is not an approved operating range.
At q=130°, the wheel disk contains A in this projection. Lateral separation
and CAD geometry are needed to decide collision. With the wheel grounded
and body pitch zero, increasing local Oᵧ **lowers** A: z_A = r − Oᵧ.

In [ ]:
checks = []
for angle in (70, 75, 90, 120, 130, 135):
    sample = sample_bench_leg(geometry, math.radians(angle))
    derivative = leg_jacobian(geometry, sample.pose).d_o
    checks.append(
        {
            "q (deg)": angle,
            "J_Ox (mm/rad)": derivative[0],
            "J_Oy (mm/rad)": derivative[1],
            "z_A at pitch 0 (mm)": geometry.wheel_radius_mm - sample.pose.o[1],
            "Projected clearance (mm)": sample.projected_hip_clearance_mm,
            "Wheel projection contains A": sample.projected_hip_clearance_mm < 0,
        }
    )
display(
    HTML(
        result_table(
            checks, caption="Geometric diagnostics · no CAD collision decision"
        )
    )
)
np.testing.assert_allclose(
    [checks[0]["J_Ox (mm/rad)"], checks[-1]["J_Oy (mm/rad)"]],
    [500, 2175],
    atol=1,
    rtol=0,
)
assert checks[-2]["Wheel projection contains A"]
fig, ax = plt.subplots(figsize=(7, 4.5), constrained_layout=True)
draw_leg_linkage(
    solve_leg_pose(geometry, math.radians(130)), geometry, ax=ax, ground_contact=False
)
ax.set_title("q = 130° · projected wheel overlaps A · check lateral CAD clearance")
plt.show()